<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/generateText/Phi_3_mini_4k(FAQ_Ds).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q bitsandbytes>=0.46.1
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
import pandas as pd
import re
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
device = torch.device("cuda")
device

device(type='cuda')

In [5]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)

In [6]:
def remove_special_tokens(text):
    text = re.sub(r'<\|.*?\|>', '', text)
    return text.strip()

In [7]:
def preProcessText(text):
  text = text.lower()
  text = re.sub(r'<.*?>', '', text)
  text = re.sub(r'http\S+|www\.\S+', '', text)
  text = re.sub(r'â€™', "'", text)
  text = re.sub(r'([!?.,])\1+', r'\1', text)
  text = re.sub(r'\s+([.,!?])', r'\1', text)
  text = re.sub(r'\s+', ' ', text).strip()
  text = re.sub(r'[^\w\s]', '', text)
  return text


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Phi-3-mini-4k-instruct",
    # "/content/drive/MyDrive/Colab Notebooks/mental-health-qa-phi3mini4k", #unsloth/Phi-3-mini-4k-instruct
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((3072,), eps=1e-05)
    

In [9]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

TokenizersBackend(name_or_path='unsloth/Phi-3-mini-4k-instruct', vocab_size=32000, model_max_length=4096, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<|placeholder2|>", rstrip=Tr

In [10]:
messages = [
    # {"role": "assistant", "content": "I feel completely lost after my dog died. What should I do to cope day to day?"},
    {"role": "user", "content": input("")}
]

How common is depression in the US?


In [11]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
	return_dict=True,
	return_tensors="pt").to(device)
inputs

{'input_ids': tensor([[32010,  1128,  3619,   338,   316,  2590,   297,   278,  3148, 29973,
         32007, 32001]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [12]:
# model_inputs = encoded_message.to(device)
# model_inputs

In [13]:
outputs = model.generate(**inputs, max_new_tokens=150, max_length=150, num_return_sequences=3, do_sample=True)
outputs

Both `max_new_tokens` (=150) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[32010,  1128,  3619,   338,   316,  2590,   297,   278,  3148, 29973,
         32007, 32001,  7579,   304,   278,  3086,  8907,   310,   341, 13703,
         15202,   313, 29940,  7833, 29950,   511,   408,   310,  1009,  1833,
         23415,   297, 29871, 29906, 29900, 29896, 29955, 29892,   385, 15899,
         29871, 29955, 29889, 29896, 29995,   310,   501, 29889, 12552,  1330,
           471,  8471,   411,   263, 19119,  4486,  2264, 29889, 10056,   292,
           393,   278, 13638,   310,  1438, 19119,  4486,  2264,   267,  1033,
          6416,  2629,   278,  6874,   310,   316,  2590, 29892,   372,   508,
           367, 28705,   393,   316,  2590,   338,   263, 16951,  3619, 19119,
          9045,  5932,   297,   278,  3303,  3900, 29889, 32007, 32009, 32009,
         32009, 32009],
        [32010,  1128,  3619,   338,   316,  2590,   297,   278,  3148, 29973,
         32007, 32001,   897,  2590,  6602, 29879, 14746,   310, 23035,  1269,
          1629, 29889,  7579

In [14]:
print(tokenizer.decode(outputs))

['<|user|> How common is depression in the US?<|end|><|assistant|> According to the National Institute of Mental Health (NIMH), as of their last reporting in 2017, an estimated 7.1% of U.limperson was living with a mental illness. Considering that the majority of these mental illnesses could fall within the scope of depression, it can be argued that depression is a significantly common mental health concern in the United States.<|end|><|placeholder6|><|placeholder6|><|placeholder6|><|placeholder6|>', '<|user|> How common is depression in the US?<|end|><|assistant|> Depression affects millions of Americans each year. According to data from the Substance Abuse and Mental Health Services Administration (SAMHSA), in 2020, an estimated 9.7% of U.ös adults experienced at least one major depressive episode. This suggests that approximately 8.5 million American adults, or one in every 14 individuals, suffered from depression.<|end|>', "<|user|> How common is depression in the US?<|end|><|assis

In [15]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Mental_Health_FAQ.csv")

In [16]:
questions = dataset['Questions'].astype("str").apply(preProcessText).apply(remove_special_tokens).values
questions[:1]

array(['what does it mean to have a mental illness'], dtype=object)

In [17]:
answers = dataset['Answers'].astype("str").apply(preProcessText).apply(remove_special_tokens).values
answers[:1]

array(['mental illnesses are health conditions that disrupt a persons thoughts emotions relationships and daily functioning they are associated with distress and diminished capacity to engage in the ordinary activities of daily life mental illnesses fall along a continuum of severity some are fairly mild and only interfere with some aspects of life such as certain phobias on the other end of the spectrum lie serious mental illnesses which result in major functional impairment and interference with daily life these include such disorders as major depression schizophrenia and bipolar disorder and may require that the person receives care in a hospital it is important to know that mental illnesses are medical conditions that have nothing to do with a persons character intelligence or willpower just as diabetes is a disorder of the pancreas mental illness is a medical condition due to the brains biology similarly to how one would treat diabetes with medication and insulin mental illness is

In [18]:
question_ids = dataset['Question_ID'].astype('int').values
question_ids[:1]

array([1590140])

In [19]:
def combineText(example):
  messages = [
      {"role": "user", "content": example['questions']},
      {"role": "assistant", "content": example['answers']}
  ]
  formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return {"text": formatted_text}

In [20]:
def encode(example):
  return tokenizer(example['text'], truncation=True, padding=True, max_length=128)

In [21]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [22]:
datasets = Dataset.from_dict({
    "questions": questions,
    "answers": answers
})
datasets

Dataset({
    features: ['questions', 'answers'],
    num_rows: 98
})

In [23]:
datasets_split = datasets.train_test_split( test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['questions', 'answers'],
        num_rows: 78
    })
    test: Dataset({
        features: ['questions', 'answers'],
        num_rows: 20
    })
})

In [24]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['questions', 'answers'],
    num_rows: 78
})

In [25]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['questions', 'answers'],
    num_rows: 20
})

In [26]:
trainSet = trainSet.map(combineText)

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

In [27]:
testSet = testSet.map(combineText)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [28]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Dataset({
    features: ['questions', 'answers', 'text', 'input_ids', 'attention_mask'],
    num_rows: 78
})

In [29]:
testSet = testSet.map(encode, batched=True)
testSet

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset({
    features: ['questions', 'answers', 'text', 'input_ids', 'attention_mask'],
    num_rows: 20
})

In [30]:
trainSet = trainSet.map(add_labels)
trainSet

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Dataset({
    features: ['questions', 'answers', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 78
})

In [31]:
testSet = testSet.map(add_labels)
testSet

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset({
    features: ['questions', 'answers', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 20
})

In [32]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=1, # Further reduced batch size to prevent OOM
    per_device_eval_batch_size=1, # Reduced eval batch size for consistency
    gradient_accumulation_steps=8, # Use gradient accumulation to achieve an effective batch size of 1 * 8 = 8
    eval_strategy='steps',
    weight_decay=0.01,
    warmup_steps=20,
    logging_dir=None,
    fp16=True,
    bf16=False, # Use bfloat16 for better memory stability and efficiency on T4
    logging_steps=50,
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    max_grad_norm = 1.0,
    report_to="none"
)

In [33]:
model.add_adapter(lora_config, adapter_name="my_adapter")

In [34]:
trainer = Trainer(
    model=model,
    args=trainingArgs,
    train_dataset=trainSet,
    eval_dataset=testSet
)

In [35]:
trainer.evaluate(testSet)

{'eval_loss': 3.791870594024658,
 'eval_model_preparation_time': 0.0164,
 'eval_runtime': 4.4787,
 'eval_samples_per_second': 4.466,
 'eval_steps_per_second': 4.466}

In [36]:
trainer.predict(testSet)

PredictionOutput(predictions=array([[[-1.03125   , -1.5683594 , -1.7324219 , ...,  0.09509277,
          0.10217285,  0.10559082],
        [-1.03125   , -1.5683594 , -1.7324219 , ...,  0.09509277,
          0.10217285,  0.10559082],
        [-1.03125   , -1.5683594 , -1.7324219 , ...,  0.09509277,
          0.10217285,  0.10559082],
        ...,
        [34.8125    , 30.390625  , 31.515625  , ..., 28.        ,
         28.015625  , 28.015625  ],
        [15.6328125 , 20.140625  , 16.984375  , ..., 10.0234375 ,
         10.0234375 , 10.0234375 ],
        [18.953125  , 15.9609375 , 15.234375  , ..., 12.1328125 ,
         12.140625  , 12.140625  ]],

       [[ 2.078125  ,  2.3007812 ,  1.1708984 , ...,  1.0029297 ,
          1.0146484 ,  1.0136719 ],
        [34.125     , 31.859375  , 35.75      , ..., 30.484375  ,
         30.5       , 30.5       ],
        [33.75      , 31.484375  , 34.96875   , ..., 29.984375  ,
         29.984375  , 29.984375  ],
        ...,
        [36.9375    , 34.

In [37]:
trainer.train()

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=4.4473121643066404, metrics={'train_runtime': 102.9426, 'train_samples_per_second': 1.515, 'train_steps_per_second': 0.194, 'total_flos': 446748504883200.0, 'train_loss': 4.4473121643066404, 'epoch': 2.0})

In [38]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/mental-health-qa-phi3mini4k")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [39]:
trainer.evaluate(testSet)

{'eval_loss': 3.748659133911133,
 'eval_model_preparation_time': 0.0164,
 'eval_runtime': 4.7162,
 'eval_samples_per_second': 4.241,
 'eval_steps_per_second': 4.241,
 'epoch': 2.0}

In [40]:
trainer.predict(testSet)

PredictionOutput(predictions=array([[[-1.03125   , -1.5683594 , -1.7324219 , ...,  0.09509277,
          0.10217285,  0.10559082],
        [-1.03125   , -1.5683594 , -1.7324219 , ...,  0.09509277,
          0.10217285,  0.10559082],
        [-1.03125   , -1.5683594 , -1.7324219 , ...,  0.09509277,
          0.10217285,  0.10559082],
        ...,
        [35.15625   , 30.8125    , 32.125     , ..., 28.265625  ,
         28.265625  , 28.265625  ],
        [16.109375  , 20.453125  , 17.34375   , ..., 10.3515625 ,
         10.3515625 , 10.3515625 ],
        [19.6875    , 16.828125  , 16.015625  , ..., 12.9296875 ,
         12.9296875 , 12.9375    ]],

       [[ 2.1386719 ,  2.3046875 ,  1.2099609 , ...,  0.9995117 ,
          1.0107422 ,  1.0107422 ],
        [34.21875   , 31.875     , 35.8125    , ..., 30.53125   ,
         30.53125   , 30.546875  ],
        [33.84375   , 31.5625    , 35.03125   , ..., 30.015625  ,
         30.03125   , 30.03125   ],
        ...,
        [37.03125   , 34.